# Distributions, measured conversions and corrections

Three things the table of medians cannot tell you, from Python: how galaxies are spread around a relation (blue cloud, green valley, red sequence), how much a change of definition moves a number in a given code, and which curves in the dataset were corrected after they were taken from a third-party compilation. Run from the repository root (or after `pip install .`). Everything below reads the files in the dataset release you have; nothing is fetched.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import sim_highline as sh
df = sh.load()
dist = sh.load_distributions()
print("dataset version", df.attrs["version"], "|", dist.record_id.nunique(), "distribution records")

## 1. The spread of specific star formation rate: green valley and red sequence

A distribution record holds the number of galaxies on a grid of stellar mass and specific SFR (or colour). The three CAMELS codes below are the fiducial model of IllustrisTNG, SIMBA and Astrid, 27 boxes each pooled. The *green valley fraction* is the share of galaxies with log sSFR between -11.0 and -10.5 (the two quenching thresholds used elsewhere in the dataset); the *quenched fraction* is the share below -11.0. Galaxies with zero SFR are counted in the lowest sSFR bin and therefore as quenched.

In [ ]:
ids = {"IllustrisTNG": "camels.tng.ssfr-dist.cv.z0", "SIMBA": "camels.simba.ssfr-dist.cv.z0", "Astrid": "camels.astrid.ssfr-dist.cv.z0"}
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for code_name, rid in ids.items():
    m, xe, ye = sh.grid(dist, rid)
    xc = 0.5 * (xe[1:] + xe[:-1])
    n = m.sum(axis=1)
    centre = 0.5 * (ye[1:] + ye[:-1])
    green = m[:, (centre > -11.0) & (centre < -10.5)].sum(axis=1) / n
    quenched = m[:, centre < -11.0].sum(axis=1) / n
    ok = n >= 30
    ax[0].plot(xc[ok], green[ok], marker="o", label=code_name)
    ax[1].plot(xc[ok], quenched[ok], marker="o", label=code_name)
    print(f"{code_name:13s} galaxies {int(n.sum()):6d}   quenched share at log M* = {xc[ok][len(xc[ok])//2]:.2f}: {quenched[ok][len(xc[ok])//2]:.2f}")
ax[0].set(xlabel="log10 M* [Msun]", ylabel="green-valley fraction"); ax[1].set(xlabel="log10 M* [Msun]", ylabel="quenched fraction (log sSFR < -11)")
ax[0].legend(); plt.tight_layout(); plt.show()

The three codes differ in how many galaxies sit between the blue cloud and the red sequence, which a median sSFR curve hides. Read the warning of each record before using the lowest-mass bins: galaxies near the resolution limit with zero instantaneous SFR count as quenched.

## 2. What a change of definition does, measured

The same galaxy has a different stellar mass if you count all bound stars, the stars within twice the half-mass radius, or those within the radius of maximum circular velocity. `measured_shift` returns the median change in dex between two definitions in one code, the 16th to 84th percentile of the galaxies, and the standard error across the 27 boxes. It refuses to extrapolate. Read `docs/CONVERSIONS.md` for what each pair means and for the limits.

In [ ]:
for code_name in ("IllustrisTNG", "SIMBA", "Astrid"):
    s = sh.measured_shift("mass.total-to-rad2", code_name, 10.0)
    print(f"{code_name:13s} stars within 2 R_half vs all bound stars at log M* = 10: {s['median']:+.3f} dex  (16-84%: {s['p16']:+.3f} to {s['p84']:+.3f}; seed-to-seed {s['se']:.3f})")

These offsets are different for different codes, which is why the Explorer does not apply a single aperture correction. Here is the largest effect in the table, the radius of maximum circular velocity in SIMBA, against the same quantity in Astrid:

In [ ]:
for code_name in ("SIMBA", "Astrid"):
    s = sh.measured_shift("mass.total-to-radmax", code_name, 10.5)
    print(code_name, f"{s['median']:+.2f} dex (16-84%: {s['p16']:+.2f} to {s['p84']:+.2f})")
# a built-in check: the stellar half-mass radius holds half of the stars by definition
print("half-mass aperture:", round(sh.measured_shift("mass.total-to-rad1", "IllustrisTNG", 10.0)["median"], 3), "expected", round(np.log10(0.5), 3))

## 3. Corrections made to third-party curves

Some observation curves come from a compilation whose conversion of the original paper's numbers did not match the paper. Each such curve is corrected in the dataset by a recorded rule: the shift, the reason and the evidence. The original compilation values are recoverable (the hash of the points as ingested is in the record's provenance).

In [ ]:
for c in sh.corrections():
    print(c["id"], "-", len(c["records"]), "records; dx =", c.get("dxDex"), "dex, dy =", c.get("dyDex"), "dex")
    print("  ", c["summary"])

## 4. Cite exactly what you used

Links, the BibTeX of every selected record and the dataset version belong together: results change when curves are corrected.

In [ ]:
sel = sh.select(df, relation="gsmf", z=4.0, kind="observation")
print(df.attrs["version"], "|", sel.record_id.nunique(), "observation curves selected")
print(sh.bibtex(sel)[:600])